# Versuch SPE: Spektroskopie

| | |
|---|---|
| **Name** | *Vorname Nachname* |
| **Matrikelnummer** | *123456* |
| **Gruppe** | *…* |
| **Betreuer:in** | *…* |
| **Versuchstag** | *TT.MM.JJJJ* |

<br>

> **Notebook als Versuchsbericht:** Sie können dieses Notebook als Versuchsbericht abgeben. Werten Sie dazu alle Ihre Messdaten gemäß den Aufgaben aus und diskutieren Sie Ihre Ergebnisse in Markdown-Zellen an geeigneter Stelle im Notebook. Die Diskussion ist ein wichtiger Teil des Berichts. Laden Sie das Notebook als `.ipynb`-Datei bis zum nächsten Termin im moodle-Lernraum hoch. Gehören weitere Dateien dazu, zum Beispiel eine CSV-Datei, aus der Sie Messdaten eingelesen haben, oder Bilder, die nicht im Notebook eingebettet sind, laden Sie alles zusammen als `.zip`-Archiv hoch. Benennen Sie die Datei nach dem Schema `Gruppe_Nachname1_Nachname2_SPE` mit der Bezeichnung Ihrer Gruppe (zum Beispiel CMo03) und den Nachnamen aller Gruppenmitglieder. Ausnahme: Gruppen mit Kurzveröffentlichung, siehe den Hinweis darunter.

> **Hinweis für Gruppen mit Kurzveröffentlichung**
>
> Ist dieser Versuch in Ihrem Terminplan mit einem Stern gekennzeichnet (SPE*), schreiben Sie statt eines Versuchsberichts eine wissenschaftliche Kurzveröffentlichung. Sie können dieses Notebook dann zur Auswertung Ihrer Messdaten verwenden, geben es aber nicht ab. Die Teile, die zum Versuchsbericht gehören, also die Angaben in der Kopftabelle und die Diskussionszellen, entfallen.

**Vorübergehend:** Im Praktikumsprofil fehlen derzeit noch Pakete, die dieses Notebook braucht. Führen Sie die folgende Zelle deshalb zu Beginn einmal aus, sie installiert die Pakete. Meldet ein späterer Import trotzdem einen Fehler, starten Sie den Kernel neu (**Kernel** → **Restart Kernel**) und führen Sie die Zellen erneut aus.

In [ ]:
%pip install --quiet numpy scipy matplotlib uncertainties odrpack==0.6.1

In diesem Notebook werden die Messdaten zum Versuch SPE ausgewertet (Anleitung Kapitel 9, Aufgaben in Abschnitt 9.5).

Es werden verwendet:

- `numpy` für Rechnungen mit Messwerten
- `matplotlib.pyplot` für Diagramme
- `odr_fit` aus `odrpack` für den linearen Fit mit Unsicherheiten in beide Richtungen

Die Auswertung besteht aus vier Teilen:

1. Beugung am Gitter und Brechung am Prisma (Aufgabe 1)
2. Emissionsspektren von Lichtquellen (Aufgabe 2)
3. Bestimmung des Extinktionskoeffizienten (Aufgabe 3)
4. Farbstoffkonzentration von Bols Blue (Aufgabe 4)

Notation wie im Versuch MEDA: $\bar x$ Mittelwert, $s_x$ Standardabweichung, $u_x$ Messunsicherheit. Ergebnisse werden in der Form $(\bar x \pm u_x)\,\mathrm{Einheit}$ angegeben. $u_x$ wird auf eine signifikante Stelle aufgerundet (auf zwei, wenn die erste Stelle eine 1 oder 2 ist), der Bestwert auf dieselbe Dezimalstelle gerundet.

Die verwendeten Python-Werkzeuge (Arrays, `plt.errorbar()`, `odr_fit()`, Kovarianzmatrix, Fehlerfortpflanzung) kennen Sie aus der Einführung PYT.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from odrpack import odr_fit

## 1. Beugung am Gitter und Brechung am Prisma

Dieser Teil enthält keine Rechnung. Beschreiben und erklären Sie Ihre Beobachtungen aus den Handexperimenten mit Prisma und Gitter (Aufgabe 1): Art der Aufspaltung (kontinuierlich oder diskret), Farbreihenfolge bei Prisma und Gitter, Ursachen der beiden Effekte (Gl. 9.8, 9.10 und 9.11).

### Aufgabe 1: Beobachtungen und Erklärung

*…*

## 2. Emissionsspektren von Lichtquellen

Beschreiben Sie die aufgenommenen Spektren der Lichtquellen (Laborlampe, LEDs, Laser) und erklären Sie Unterschiede und Gemeinsamkeiten (Aufgabe 2).

Fügen Sie das Overlay-Diagramm aus Spectra Suite als Bild ein. Alternativ können die gespeicherten Spektren mit `np.genfromtxt()` eingelesen und mit `plt.plot()` dargestellt werden.

### Aufgabe 2: Beschreibung und Vergleich der Spektren

*…*

## 3. Bestimmung des Extinktionskoeffizienten

In diesem Teil wird aus den bei $\lambda = 670\,\mathrm{nm}$ gemessenen Intensitäten der Verdünnungsreihe der Extinktionskoeffizient $\alpha$ des Farbstoffs bestimmt.

Die Extinktion folgt aus der Intensität $I_0$ (Küvette mit Wasser) und $I_1$ (mit Farbstoff) (Gl. 9.4):

$$
E = \log_{10}\frac{I_0}{I_1}
$$

Es gilt (Gl. 9.7):

$$
E = \alpha\, c\, l
$$

Wird $E$ gegen $c$ aufgetragen, ist die Steigung der Ausgleichsgeraden $\alpha l$.

Zuerst werden die Messwerte definiert und um das Dunkelspektrum korrigiert. Danach werden Extinktionen, Unsicherheiten, Fit und Extinktionskoeffizient berechnet.

### Messwerte und Konstanten definieren

Hier werden die Konzentrationen, die bei $670\,\mathrm{nm}$ abgelesenen Intensitäten (in counts), das Dunkelspektrum und die Dicke der Küvette definiert.

Vorgaben aus der Anleitung: Stammlösung $c_0 = (0{,}064 \pm 0{,}004)\,\mathrm{g/L}$ und Verdünnungsreihe $0{,}002$ bis $0{,}032\,\mathrm{g/L}$ (Abschnitt 9.4.5), Küvettendicke $l = 1\,\mathrm{cm}$ (Aufgabe 3).

Benötigt werden:

- das Dunkelspektrum bei $670\,\mathrm{nm}$ (Lampe aus),
- die fünf Wiederholungsmessungen der Intensität bei fester Wellenlänge (Abschnitt 9.4.3),
- die Intensität $I_0$ der Küvette mit reinem Wasser,
- die Intensitäten $I_1$ der Verdünnungsreihe,
- die Intensität der Küvette mit unverdünntem Bols Blue (für Teil 4).

Die Unsicherheit der Intensität ist die Standardabweichung der fünf Wiederholungsmessungen bei fester Wellenlänge (`np.std(..., ddof=1)`). Sie gilt für jede Intensitätsmessung.

Die Konzentrationen werden in $\mathrm{g/L}$ und $l$ in $\mathrm{m}$ angegeben, damit $\alpha$ die Einheit $\mathrm{L/(g\,m)}$ erhält.

In [ ]:
# Ihr Code hier


### Dunkelspektrum abziehen

Der Mittelwert des Dunkelspektrums $I_\mathrm{offset}$ wird mit `np.mean()` berechnet und von allen gemessenen Intensitäten abgezogen (Aufgabe 2):

$$
I = I_\mathrm{roh} - I_\mathrm{offset}
$$

In [ ]:
# Ihr Code hier


### Aufgabe 3: Extinktion und Unsicherheit berechnen

Für jede Konzentration wird die Extinktion berechnet (Gl. 9.4):

$$
E = \log_{10}\frac{I_0}{I_1} = \log_{10}(I_0) - \log_{10}(I_1)
$$

Der dekadische Logarithmus wird mit `np.log10()` berechnet.

Die Unsicherheit von $E$ folgt aus Gaußscher Fehlerfortpflanzung über $I_0$ und $I_1$ (beide mit der Unsicherheit $u_I$):

$$
u_E =
\sqrt{
\left(
\frac{\partial E}{\partial I_0}\,u_I
\right)^2
+
\left(
\frac{\partial E}{\partial I_1}\,u_I
\right)^2
}
$$

Leiten Sie $E(I_0, I_1)$ nach $I_0$ und nach $I_1$ ab. Für die Ableitung des dekadischen Logarithmus gilt

$$
\frac{\mathrm{d}}{\mathrm{d}x}\log_{10}(x) = \frac{1}{x \ln(10)}
$$

Der natürliche Logarithmus wird mit `np.log()` berechnet.

**Hinweis:** Die Fehlerfortpflanzung kann auch mit dem Paket `uncertainties` berechnet werden. Dazu werden die Messgrößen zusammen mit ihren Unsicherheiten als Arrays angelegt, zum Beispiel `I0_u = unp.uarray(I0, u_I)` und `I1_u = unp.uarray(I1, u_I)` (nach `import uncertainties.unumpy as unp`). Mit diesen Arrays wird wie gewohnt gerechnet, Funktionen wie Logarithmus oder Wurzel werden dabei als `unp.log10()` und `unp.sqrt()` geschrieben. Bestwert und Unsicherheit des Ergebnisses liefern `unp.nominal_values(E)` und `unp.std_devs(E)`. Achten Sie auf die richtige Angabe signifikanter Stellen. `uncertainties` rundet Unsicherheiten kaufmännisch statt aufzurunden.

In [ ]:
# Ihr Code hier


### Unsicherheit der Konzentrationen berechnen

Alle Konzentrationen der Verdünnungsreihe wurden aus der Stammlösung $c_0$ hergestellt (Abschnitt 9.4.5). Überlegen Sie, wie sich die Unsicherheit $u_{c_0}$ auf die verdünnten Lösungen auswirkt (Aufgabe 3), und berechnen Sie die Unsicherheiten $u_c$ der Konzentrationen.

Diese Unsicherheiten werden im Diagramm als Fehlerbalken in $x$-Richtung dargestellt.

In [ ]:
# Ihr Code hier


### Diagramm der Messwerte erstellen

Vor dem Fit werden die Messwerte dargestellt: $E$ gegen $c$ mit Fehlerbalken $u_c$ in $x$-Richtung und $u_E$ in $y$-Richtung. Prüfen Sie am Verlauf, ob der lineare Zusammenhang nach Gl. 9.7 vorliegt.

Zur Erinnerung: `plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")` zeichnet die Messwerte mit Fehlerbalken.

Achsen werden in der Form „Größe in Einheit“ beschriftet. $E$ ist einheitenlos.

In [ ]:
# Ihr Code hier


### Linearen Fit durchführen

Die Ausgleichsgerade hat die Form:

$$
E = m\,c + b
$$

Aus dem Vergleich mit $E = \alpha l\, c$ folgt:

$$
m = \alpha l
$$

Weil beide Größen Unsicherheiten tragen (Aufgabe 3), wird der Fit mit `odr_fit` durchgeführt. Er gewichtet die Messwerte mit ihren Unsicherheiten in beide Richtungen, genau wie die Methode minimaler und maximaler Steigung aus Abschnitt 1.3.2 die Fehlerbalken in $x$- und $y$-Richtung verwendet.

Zur Erinnerung: ODR berücksichtigt Unsicherheiten in $x$- und $y$-Richtung. Der Fit läuft in vier Schritten.

**1. Modellfunktion definieren.** Die Parameter stehen gemeinsam im Array `beta`. Die Funktion bekommt zuerst `x` und danach `beta`:

```python
def linear_odr(x, beta):
    return beta[0] * x + beta[1]
```

**2. Gewichte berechnen.** `odr_fit` erwartet die Unsicherheiten nicht direkt, sondern als Gewichte. Das Gewicht eines Messwerts ist eins geteilt durch das Quadrat seiner Unsicherheit:

$$
w_x = \frac{1}{u_x^2}
\qquad
w_y = \frac{1}{u_y^2}
$$

**3. Fit durchführen.** Übergeben werden die Modellfunktion, die Messwerte, die Startwerte `beta0` und die Gewichte:

```python
output = odr_fit(linear_odr, x, y, beta0=[1, 0], weight_x=1/u_x**2, weight_y=1/u_y**2)
```

**4. Ergebnisse auslesen.** Die Parameter stehen in `output.beta`. Ihre Unsicherheiten stehen wie bei `curve_fit` auf der Diagonalen der Kovarianzmatrix, hier heißt sie `output.cov_beta`:

```python
u_m = np.sqrt(output.cov_beta[0, 0])   # Steigung
u_b = np.sqrt(output.cov_beta[1, 1])   # Achsenabschnitt
```

**Wichtig:** `beta0` sind die Startwerte. `odr_fit` arbeitet mit relativen Schrittweiten und braucht sie in der richtigen Größenordnung, bei sehr kleinen Zahlen darf kein Startwert genau null sein.

In [ ]:
# Ihr Code hier


### Extinktionskoeffizient berechnen

Aus der Steigung folgt der Extinktionskoeffizient:

$$
\alpha = \frac{m}{l}
$$

Die Unsicherheiten der Intensitäten und der Konzentrationen stecken bereits in $u_m$. Bestimmen Sie daraus die Unsicherheit $u_\alpha$.

In [ ]:
# Ihr Code hier


### Diagramm der Regression erstellen

Die Fitgerade wird in das Diagramm der Messwerte eingezeichnet.

Zur Erinnerung: `plt.plot(x, linear_odr(x, output.beta), label="Fit")` zeichnet die Fitgerade an den $x$-Werten der Messung.

In [ ]:
# Ihr Code hier


> **Ergebnis Teil 3**
>
> Steigung $m = (\ldots \pm \ldots)\,\mathrm{L/g}$, Achsenabschnitt $b = \ldots \pm \ldots$
>
> Extinktionskoeffizient bei 670 nm: $\alpha = (\ldots \pm \ldots)\,\mathrm{L/(g\,m)}$

### Diskussion Teil 3

*…*

## 4. Farbstoffkonzentration von Bols Blue

In diesem Teil wird aus der Intensität der Küvette mit unverdünntem Bols Blue und dem Extinktionskoeffizienten aus Teil 3 die Konzentration des Farbstoffs bestimmt.

Formen Sie dazu das Lambert-Beersche Gesetz (Gl. 9.5) nach $c$ um (Aufgabe 4). Die Extinktion von Bols Blue ist (Gl. 9.4):

$$
E_\mathrm{Bols} = \log_{10}\frac{I_0}{I_\mathrm{Bols}}
$$

### Aufgabe 4: Konzentration und Unsicherheit berechnen

Zuerst wird die Extinktion $E_\mathrm{Bols}$ mit ihrer Unsicherheit wie in Teil 3 berechnet.

Die Unsicherheit der Konzentration folgt aus Gaußscher Fehlerfortpflanzung über $E_\mathrm{Bols}$ und $\alpha$:

$$
u_{c} =
\sqrt{
\left(
\frac{\partial c}{\partial E}\,u_E
\right)^2
+
\left(
\frac{\partial c}{\partial \alpha}\,u_\alpha
\right)^2
}
$$

Leiten Sie $c(E, \alpha)$ nach $E$ und nach $\alpha$ ab und setzen Sie $E_\mathrm{Bols}$, $\alpha$ sowie $u_{E}$ und $u_\alpha$ ein.

Das Ergebnis wird in $\mathrm{g/L}$ und in $\mathrm{mg/L}$ ausgegeben.

In [ ]:
# Ihr Code hier


> **Ergebnis Teil 4**
>
> $E_\mathrm{Bols} = \ldots \pm \ldots$
>
> $c_\mathrm{Bols} = (\ldots \pm \ldots)\,\mathrm{g/L} = (\ldots \pm \ldots)\,\mathrm{mg/L}$

### Diskussion Teil 4

*…*

## 5. Zusammenfassung der Ergebnisse

Zum Schluss werden die wichtigsten Ergebnisse des gesamten Versuchs gesammelt ausgegeben.

Die Zusammenfassung enthält:

1. die Steigung der Ausgleichsgeraden und den Extinktionskoeffizienten bei $670\,\mathrm{nm}$,

$$
\alpha \pm u_\alpha
$$

2. die Extinktion und die Farbstoffkonzentration von Bols Blue,

$$
c_\mathrm{Bols} \pm u_{c}
$$

Achten Sie darauf, alle Ergebnisse mit passenden Einheiten auszugeben. Tragen Sie die Ergebnisse in die Ergebnis-Zelle darunter ein und achten Sie dabei auf die richtige Angabe signifikanter Stellen.

In [ ]:
# Ihr Code hier


> **Ergebnis (Zusammenfassung)**
>
> $\alpha = (\ldots \pm \ldots)\,\mathrm{L/(g\,m)}$ bei 670 nm (Steigung $m = (\ldots \pm \ldots)\,\mathrm{L/g}$)
>
> $c_\mathrm{Bols} = (\ldots \pm \ldots)\,\mathrm{g/L} = (\ldots \pm \ldots)\,\mathrm{mg/L}$

## 6. Diskussion

*…*

## Abgabe als Versuchsbericht

Wenn Sie dieses Notebook als Versuchsbericht abgeben, prüfen Sie vorher, dass alle Aufgaben ausgewertet und Ihre Ergebnisse in Markdown-Zellen diskutiert sind, und füllen Sie die Kopfzelle aus. Laden Sie das Notebook als `.ipynb`-Datei bis zum nächsten Termin im moodle-Lernraum hoch. Gehören weitere Dateien dazu, zum Beispiel eine CSV-Datei, aus der Sie Messdaten eingelesen haben, oder Bilder, die nicht im Notebook eingebettet sind, laden Sie das Notebook zusammen mit diesen Dateien als `.zip`-Archiv hoch. Benennen Sie die Datei nach dem Schema `Gruppe_Nachname1_Nachname2_SPE` mit der Bezeichnung Ihrer Gruppe (zum Beispiel CMo03) und den Nachnamen aller Gruppenmitglieder. Bei einer Kurzveröffentlichung entfällt die Abgabe des Notebooks (Hinweis am Anfang).